In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
sys.path.append("/Workspace/Users/mandu543@gmail.com/databricks-movies-analytics/Movies_Project")

from src.utils import *

# Liste tous les schemas du catalog 'workspace'
spark.sql("SHOW SCHEMAS IN workspace").show()

# Create schema if not exists
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {GOLD_TABLE.split('.')[1]}"
)

print(f'✅ Schema {GOLD_TABLE.split('.')[1]} created or already exists')


In [0]:
import inspect
functions = [name for name,obj in globals().items() if inspect.isfunction(obj)]
# print(functions)

In [0]:
# Liste des tables Silver
df_silver_fait_film = getTable(spark,SILVER_ZONE,"silver_tmdb_movies")
display(df_silver_fait_film)

df_silver_rel_genre = getTable(spark,SILVER_ZONE,"silver_tmdb_movies_genre")
display(df_silver_rel_genre)

df_silver_rel_pays_prod = getTable(spark,SILVER_ZONE,"silver_tmdb_movies_production_country")
display(df_silver_rel_pays_prod)

df_silver_rel_compagnie_prod = getTable(spark,SILVER_ZONE,"silver_tmdb_movies_production_companie")
display(df_silver_rel_compagnie_prod)

df_silver_rel_langage_traduit = getTable(spark,SILVER_ZONE,"silver_tmdb_movies_spoken_language")
display(df_silver_rel_langage_traduit)

df_silver_dim_genre = getTable(spark,SILVER_ZONE,"dim_film_genre")
display(df_silver_dim_genre)

df_silver_dim_pays_prod = getTable(spark,SILVER_ZONE,"dim_film_production_country")
display(df_silver_dim_pays_prod)

df_silver_dim_compagnie_prod = getTable(spark,SILVER_ZONE,"dim_film_production_companie")
display(df_silver_dim_compagnie_prod)

df_silver_dim_langage_traduit= getTable(spark,SILVER_ZONE,"dim_film_spoken_language")
display(df_silver_dim_langage_traduit)



    

In [0]:
%sql
use catalog `workspace`; select count(*) from `02_silver`.`silver_tmdb_movies`;

In [0]:
# Table de fait 
table_fact_movies = GOLD_ZONE + ".fact_movies"

# Table de dimension
table_dim_genre = GOLD_ZONE + ".dim_movies_genre"
table_dim_pays_prod = GOLD_ZONE + ".dim_movies_pays_prod"
table_dim_compagnie_prod = GOLD_ZONE + ".dim_movies_compagnie_prod"
table_dim_langage_traduit = GOLD_ZONE + ".dim_movies_langage_traduit"

# Table relationnelle
table_rel_movies_genre = GOLD_ZONE + ".rel_movies_genre"
table_rel_movies_pays_prod = GOLD_ZONE + ".rel_movies_pays_prod"
table_rel_movies_compagnie_prod = GOLD_ZONE + ".rel_movies_compagnie_prod"
table_rel_movies_language_traduit = GOLD_ZONE + '.rel_movies_language_traduit'


# Créer ou maj des tables GOLD
# -----------------------------
#spark.sql(f"TRUNCATE TABLE {table_fact_movies}")

# Fait
fact_movies = insert_new_rows(spark,df_silver_fait_film,table_fact_movies)
display(fact_movies)

# Dimension
dim_movies_genre = insert_new_rows(spark,df_silver_dim_genre,table_dim_genre)
display(dim_movies_genre)

dim_movies_pays_prod = insert_new_rows(spark,df_silver_dim_pays_prod,table_dim_pays_prod)
display(dim_movies_pays_prod)

dim_movies_compagnie_prod = insert_new_rows(spark,df_silver_dim_compagnie_prod,table_dim_compagnie_prod)
display(dim_movies_compagnie_prod)

dim_movies_langage_traduit = insert_new_rows(spark,df_silver_dim_langage_traduit,table_dim_langage_traduit)
display(dim_movies_langage_traduit)

# Table relationnelle
rel_movies_genre = insert_new_rows(spark,df_silver_rel_genre,table_rel_movies_genre)
display(rel_movies_genre)

rel_movies_pays_prod = insert_new_rows(spark,df_silver_rel_pays_prod,table_rel_movies_pays_prod)
display(rel_movies_pays_prod)

rel_movies_compagnie_prod = insert_new_rows(spark,df_silver_rel_compagnie_prod,table_rel_movies_compagnie_prod)
display(rel_movies_compagnie_prod)

rel_movies_language_traduit = insert_new_rows(spark,df_silver_rel_langage_traduit,table_rel_movies_language_traduit)
display(rel_movies_language_traduit)




In [0]:
%sql
use catalog `workspace`; select count(*) from `03_gold`.`fact_movies` limit 100;

In [0]:
%sql
-- Création ou remplacement de la table dimension date
CREATE OR REPLACE TABLE `03_gold`.dim_date AS

-- Étape 1 : récupérer les bornes min et max dans la table source
WITH bounds AS (
  SELECT
    MIN(release_date) AS min_date,   -- Date minimale présente
    MAX(release_date) AS max_date    -- Date maximale présente
  FROM `03_gold`.fact_movies                -- Table source

)

-- Étape 2 : génération des lignes de dates entre min et max
SELECT
  d AS date,                       -- Date complète
  date_format(d, 'yyyyMMdd') AS date_key,
  -- Composants de date utiles pour l'analyse
  year(d) AS year,
  quarter(d) AS quarter,
  month(d) AS month,
  day(d) AS day,

  -- Informations calendaires
  dayofweek(d) AS day_of_week,
  weekofyear(d) AS week_of_year,

  -- Noms lisibles
  date_format(d, 'MMMM') AS month_name,
  date_format(d, 'EEEE') AS day_name,

  -- Indicateur weekend
  CASE
    WHEN dayofweek(d) IN (1,7) THEN true
    ELSE false
  END AS is_weekend

FROM bounds

-- Étape 3 : création d'une séquence de dates entre min_date et max_date
-- sequence() produit un tableau de dates
-- explode() transforme ce tableau en lignes
LATERAL VIEW explode(
  sequence(min_date, max_date, interval 1 day)
) t AS d;
 

In [0]:
"""
# -----------------------------
# Créer ou remplacer la table Gold en filtrant certaines colonnes
# -----------------------------
 

try:
    # Lire la table Silver
    df_silver = spark.table(SILVER_TABLE)
    
    # On supprime les colonnes inutiles pour obtenir un DataFrame plus propre
    df_gold = df_silver.drop("backdrop_path", "imdb_id", "poster_path", "keywords")
    
    # Écrire dans la table Gold
    df_gold.write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE)
    
    # Optimiser pour BI
    spark.sql(f"OPTIMIZE {GOLD_TABLE} ZORDER BY release_date")
    
    print(f"✅ Gold table {GOLD_TABLE} ready with selected columns")

except Exception as e:
    print(f"❌ Erreur lors de la création de la table Gold : {e}")


"""

